# AIC2026 TEAM-EVAL — Exact Dense Raw Renderer

Required Kaggle inputs:
1. Raw AIC corpus: `/kaggle/input/datasets/nadkli/dataset-aic` (nested roots supported).
2. Offline current repository snapshot containing `src/aic2026_eval`: `/kaggle/input/datasets/irthn1311/aic2026-team-eval-repository`.
3. AI-authored request dataset containing exactly one `anchor_requests.jsonl`: `/kaggle/input/datasets/irthn1311/aic2026-team-eval-anchor-requests` (nested root supported).

Internet required: **No**. No model asset is required. Output ZIP: `/kaggle/working/aic2026_team_eval_dense_bundle.zip`.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from zipfile import ZipFile
DATA_INPUT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
REPO_INPUT = Path(os.environ.get('AIC_REPO_ROOT', '/kaggle/input/datasets/irthn1311/aic2026-team-eval-repository'))
ANCHOR_INPUT = Path(os.environ.get('AIC_ANCHOR_REQUEST_ROOT', '/kaggle/input/datasets/irthn1311/aic2026-team-eval-anchor-requests'))
OUTPUT_ROOT = Path('/kaggle/working/aic2026_team_eval_dense')
ZIP_PATH = Path('/kaggle/working/aic2026_team_eval_dense_bundle.zip')
print({'data_input': str(DATA_INPUT), 'repo_input': str(REPO_INPUT), 'anchor_input': str(ANCHOR_INPUT), 'output_zip': str(ZIP_PATH), 'internet_required': False})

In [ ]:
sys.path.insert(0, str(REPO_INPUT / 'src')) if (REPO_INPUT / 'src/aic2026_eval').is_dir() else None
try:
    from aic2026_eval.discovery import resolve_dataset_root, resolve_named_file, resolve_repository_root
except ModuleNotFoundError:
    if not REPO_INPUT.exists(): raise RuntimeError(f'Required offline repository dataset is missing: {REPO_INPUT}')
    markers = list(REPO_INPUT.rglob('src/aic2026_eval/discovery.py'))
    if len(markers) != 1: raise RuntimeError(f'Expected one nested TEAM-EVAL repository marker under {REPO_INPUT}; found {markers}')
    sys.path.insert(0, str(markers[0].parents[2] / 'src'))
    from aic2026_eval.discovery import resolve_dataset_root, resolve_named_file, resolve_repository_root
REPO_ROOT = resolve_repository_root(REPO_INPUT)
DATASET_ROOT = resolve_dataset_root(DATA_INPUT)
ANCHOR_REQUESTS = resolve_named_file(ANCHOR_INPUT, 'anchor_requests.jsonl')
sys.path.insert(0, str(REPO_ROOT / 'src'))
commit_result = subprocess.run(['git','rev-parse','HEAD'], cwd=REPO_ROOT, capture_output=True, text=True, check=False)
BUILD_COMMIT = commit_result.stdout.strip() if commit_result.returncode == 0 else os.environ.get('AIC_BUILD_COMMIT', 'UNKNOWN_OFFLINE_SNAPSHOT')
print({'resolved_repo': str(REPO_ROOT), 'resolved_dataset': str(DATASET_ROOT), 'resolved_anchor_requests': str(ANCHOR_REQUESTS), 'commit': BUILD_COMMIT})

In [ ]:
from aic2026_eval.io import read_jsonl
REQUESTS = read_jsonl(ANCHOR_REQUESTS)
assert REQUESTS, 'anchor_requests.jsonl must not be empty'
print('anchor_count:', len(REQUESTS), 'modes:', sorted({str(row.get('mode','')).upper() for row in REQUESTS}))

In [ ]:
from aic2026_eval.pipeline import run_dense
RESULT = run_dense(dataset_root=DATASET_ROOT, repository_root=REPO_ROOT, anchor_requests_path=ANCHOR_REQUESTS, output_root=OUTPUT_ROOT, build_commit=BUILD_COMMIT)
assert RESULT['DENSE_RENDERER_STATUS'] == 'READY' and RESULT['rendered_anchor_count'] == len(REQUESTS)
manifest = read_jsonl(OUTPUT_ROOT / 'dense_manifest.jsonl')
assert all(row['frame_identity_exact'] and row['requested_frame_ids'] == row['actual_frame_ids'] for row in manifest)
print(json.dumps(RESULT, indent=2))

In [ ]:
assert Path(RESULT['zip_path']) == ZIP_PATH and ZIP_PATH.is_file()
with ZipFile(ZIP_PATH) as archive: members = archive.namelist()
assert not any(name.endswith(('.mp4','.npy','.npz','.pt','.pth')) for name in members)
print('DENSE_RENDERER_STATUS=READY')
print('SEMANTIC_JUDGMENT_PERFORMED=NO')
print('DOWNLOAD ZIP:', ZIP_PATH, 'size_bytes=', ZIP_PATH.stat().st_size, 'members=', len(members))